In [ ]:
# =============================================================================
# AUTOENCODERS & VARIATIONAL AUTOENCODERS (VAE)
# =============================================================================
#
# ---------------------------------------------------------------------------
# WHERE ARE WE ON THE PATH?
# ---------------------------------------------------------------------------
# Week2: GPT-style models that PREDICT THE NEXT TOKEN (language).
# Week3 so far: RAG, Whisper fine-tuning — use / adapt existing models.
# THIS NOTEBOOK: a different family — models that COMPRESS then REBUILD.
#
# Easy split:
#   GPT  = "finish the sentence"
#   AE   = "squeeze this into a tiny code, then rebuild the original"
#
#
# ---------------------------------------------------------------------------
# WHAT IS AN AUTOENCODER? (picture first)
# ---------------------------------------------------------------------------
# Think of packing a suitcase:
#
#   ORIGINAL (big)          LATENT (tiny code)         RECONSTRUCTION
#   ┌─────────────────┐         ┌─────┐         ┌─────────────────┐
#   │  image / audio  │ ──enc──▶│ z   │──dec──▶ │  almost the     │
#   │  / sentence     │         │     │         │  same thing     │
#   └─────────────────┘         └─────┘         └─────────────────┘
#        encoder                  bottleneck           decoder
#     (compress)               "the essence"         (decompress)
#
# Training job: make reconstruction ≈ original.
# Loss is usually "how different is rebuild vs input?"
#   images → pixel MSE / MAE
#   vectors → MSE
#
# Sticky: the useful part is often NOT the rebuild — it's the tiny code z.
#   z is a compressed fingerprint of the input.
#
#
# ---------------------------------------------------------------------------
# WHY A BOTTLENECK?
# ---------------------------------------------------------------------------
# If z is as big as the input, the net can cheat: copy everything.
# Make z SMALL → the model MUST throw away noise and keep patterns.
#
# Analogy: summarize a 10-page call into 8 numbers, then try to rewrite
# the page. Those 8 numbers had better store what actually mattered.
#
#
# ---------------------------------------------------------------------------
# DEEP BUT EASY: what the two halves learn
# ---------------------------------------------------------------------------
# Encoder:  x  →  z     "what is the gist of this example?"
# Decoder:  z  →  x̂     "given only the gist, redraw / replay it"
#
# Under the hood: stacked Linear (or conv) layers.
#   encoder:  wide → narrower → z
#   decoder:  z → wider → original size
#
# Same gradient idea as MiniGPT: loss.backward() nudges knobs so x̂ matches x.
#
#
# ---------------------------------------------------------------------------
# WHAT IS A VAE? (autoencoder + a little probability)
# ---------------------------------------------------------------------------
# Plain AE: z is one exact point ("this image = this coordinate").
#
# VAE: z is a SMALL CLOUD, not a pin.
#   Encoder outputs TWO things per input:
#     μ (mu)    = center of the cloud  ("typical code for this example")
#     σ (sigma) = how spread out       ("how unsure / how much wiggle")
#   Then we SAMPLE z from that cloud:  z ≈ μ + σ * noise
#   Decoder rebuilds from the sample.
#
# Picture:
#
#   AE:   x ──▶  • z  ──▶ x̂
#
#   VAE:  x ──▶  (μ, σ)  ──▶ sample a point in the blob ──▶ x̂
#                  ☁
#
# Why bother?
#   1) You can SAMPLE NEW z's and decode → generate new-looking examples
#      (not only reconstruct training items).
#   2) Nearby z's decode to similar things → a smooth "map" of the data.
#
# Extra VAE loss (KL term), easy English:
#   "Don't let each cloud fly to a random galaxy. Keep clouds near a
#    simple standard blob (usually a normal bump at 0). Then random
#    samples from that blob still decode to realistic stuff."
#
# Sticky:
#   AE  = compressor / fingerprint machine
#   VAE = compressor that also knows how to DREAM new examples
#
#
# ---------------------------------------------------------------------------
# AE vs VAE vs GPT (don't mix them)
# ---------------------------------------------------------------------------
#   GPT:  given past tokens → next token     (language modeling)
#   AE:   given x → rebuild x                (reconstruction)
#   VAE:  given x → distribution over z → rebuild x  (+ can sample new x)
#
#
# ---------------------------------------------------------------------------
# CONNECTION TO YOUR EARPIECE / SALES-COACH PRODUCT
# ---------------------------------------------------------------------------
# You may not ship a VAE as the whisper engine (that's still ASR + LLM).
# Where this family shows up around the product:
#   - Audio / spectrogram autoencoders: compact speech fingerprints
#   - Embedding spaces: similar calls sit near each other (search, RAG)
#   - Anomaly: if reconstruction error is huge, the audio/text looks "weird"
#   - Generative cousins: later diffusion / audio codecs use encoder–decoder
#     thinking (compress → generate)
#
# Today's goal: understand encoder, bottleneck z, decoder, and why VAE
# samples instead of using one exact z.
#
# Next cells (typical): tiny AE on simple vectors/images → add VAE sampling.
